In [ ]:
# 식품 영양 정보를 가져와서 csv파일로 저장 
# 가져올 정보. 식품명, 탄수화물, 지방, 단백질, 나트륨, 총칼로리 

# 필요한 패키지
import requests
import pandas as pd

In [ ]:
# 공공데이터 url = End Point + 상세기능 url
url = 'https://apis.data.go.kr/1471000/FoodNtrCpntDbInfo02/getFoodNtrCpntDbInq02'

serviceKey = 'yokZLTATxLSYExFfHZfEWjWDoB6g9Q0fIcgD3rXjIO0bvc7AB5H5QIDdggjBlOqvGMj7ouM7SWGAU24hcvDOEQ=='
page = 1

params = {
	'serviceKey' : serviceKey, # 필수, 나머지는 선택
	'pageNo' : page, 
	'numOfRows' : 10,
	'type' : 'json', 
}

response = requests.get(url, params=params)

try:
	response.raise_for_status()

except Exception as e:
	print(f'예외 발생 : {e}')

In [ ]:
items = response.json()['body']['items']

list = [] # 식품 정보를 관리할 리스트

for item in items:
	# 식품명, 탄수화물, 지방, 단백질, 나트륨, 총칼로리 
	dic = {
		'식품명' : item['FOOD_NM_KR'],
		'탄수화물(g)': item['AMT_NUM6'],
		'지방(g)'   : item['AMT_NUM4'], 
		'단백질(g)' : item['AMT_NUM3'],
		'나트륨(mg)' : item['AMT_NUM13'],
		'총칼로리(kal)': item['AMT_NUM1'], 
		'기준량': item['SERVING_SIZE'],
	}
	list.append(dic)

# 받아온 리스트를 데이터프레임으로 변환
df = pd.DataFrame(list)
	

In [ ]:
# 중복제거
# df = df.drop_duplicates() # 행 전체 값이 중복되면 제거
# df = df.drop_duplicates(subset=['식품명']) # 식품명 열이 중복된 경우만 제거
df = df.drop_duplicates(subset=['식품명'], keep='first') # 중복되면 첫번째를 살리고 나머지 제거
# df = df.drop_duplicates(subset=['식품명'], keep='last') # 중복되면 마지막을 살리고 나머지 제거
# df = df.drop_duplicates(subset=['식품명'], keep=False) # 중복되면 모두 제거

# 결측치 값 변경
df['탄수화물(g)'] = df['탄수화물(g)'].fillna(0) # 탄수화물(g)에 결측치가 있으면 0으로 변경

# 결측치 제거
df = df.dropna() # 행에 결측치가 하나라도 있으면 삭제
df = df.dropna(subset=['식품명']) # 식품명에 결측치가 있으면 삭제



In [ ]:
serviceKey = 'yokZLTATxLSYExFfHZfEWjWDoB6g9Q0fIcgD3rXjIO0bvc7AB5H5QIDdggjBlOqvGMj7ouM7SWGAU24hcvDOEQ=='

def get_food_nutrient(serviceKey:str, page:int=1)->pd.DataFrame:
	"""공공데이터에서 식품영양소 db 정보를 가져와서 데이터 프레임으로 반환하는 함수"""
	url = 'https://apis.data.go.kr/1471000/FoodNtrCpntDbInfo02/getFoodNtrCpntDbInq02'

	params = {
		'serviceKey' : serviceKey, # 필수, 나머지는 선택
		'pageNo' : page, 
		'numOfRows' : 10,
		'type' : 'json', 
	}

	response = requests.get(url, params=params)

	try:
		response.raise_for_status()

		items = response.json()['body']['items']

		list = [] # 식품 정보를 관리할 리스트
		labels = {
			'식품명' : 'FOOD_NM_KR',
			'탄수화물(g)':'AMT_NUM6',
			'지방(g)'   :'AMT_NUM4',
			'단백질(g)' :'AMT_NUM3',
			'나트륨(mg)' :'AMT_NUM13',
			'총칼로리(kal)':'AMT_NUM1',
			'기준량': 'SERVING_SIZE',
		}
		for item in items:
			# 식품명, 탄수화물, 지방, 단백질, 나트륨, 총칼로리 
			dic = {}
			for column, label in labels.items():
				dic[column] = item[label]
			list.append(dic)

		# 받아온 리스트를 데이터프레임으로 변환
		return pd.DataFrame(list)
	except Exception as e:
		print(f'예외 발생 : {e}')
		return pd.DataFrame()

In [ ]:
def clean_food_nutrient(df:pd.DataFrame)->pd.DataFrame:
	"""식품영양소 df에서 결측치 제거, 중복 제거하는 함수"""
	# 중복제거
	df = df.drop_duplicates(subset=['식품명'], keep='first') # 중복되면 첫번째를 살리고 나머지 제거

	# 결측치 값 변경
	df.loc[ : , '탄수화물(g)'] = df['탄수화물(g)'].fillna(0) # 탄수화물(g)에 결측치가 있으면 0으로 변경

	# 결측치 제거
	df = df.dropna(subset=['식품명']) # 식품명에 결측치가 있으면 삭제
	return df

In [ ]:
food_df = get_food_nutrient(serviceKey, 1)

food_df.loc[len(food_df)] = {'식품명' : '국밥_돼지머리'}

food_df = clean_food_nutrient(food_df)
print(food_df)